![imagenes](logo.png)

# Máquinas de soporte vectorial (SVM)

Imaginemos que tenemos varios grupos de puntos en un espacio de características. Cada punto representa un individuo descrito por sus variables, y cada color representa su clase.
El objetivo de una máquina de soporte vectorial es trazar fronteras entre esas clases, pero no de cualquier manera: las quiere tan alejadas como sea posible de los puntos de entrenamiento.

En el caso más simple —dos clases linealmente separables— bastaría con buscar una recta (o un hiperplano) que divida ambos conjuntos sin errores. Pero el SVM va más allá: no se conforma con “cualquier” recta, sino que elige aquella que maximiza la distancia entre los grupos, es decir, aquella que deja el mayor margen posible entre las observaciones de cada clase y la frontera de separación.

<img src="im017.png" width="50%" style="display: block; margin-left: auto; margin-right: auto;">

## Margen y vectores de soporte

Sea un conjunto de entrenamiento con observaciones $x_i \in \mathbb{R}^p$ y etiquetas $y_i \in \{-1, +1\}$.  Buscamos un hiperplano de la forma  

$$ w \cdot x + b = 0, $$  

tal que:  

$$ y_i(w \cdot x_i + b) \geq 1 \quad \text{para todos los } i. $$  

El ancho del margen entre ambas clases es $\frac{2}{\|w\|}$.  

Por tanto, **maximizar el margen** equivale a **minimizar** $\frac{1}{2}\|w\|^2$, sujeto a las restricciones anteriores.  

Las observaciones que *tocan* el margen -es decir, las que cumplen la igualdad $y_i(w \cdot x_i + b) = 1$- se llaman **vectores de soporte**.  
Ellas son las verdaderamente decisivas: el resto de los puntos podría desaparecer y el hiperplano seguiría igual.

En el caso lineal, la frontera es un hiperplano. El **ancho del margen** es inversamente proporcional a la norma del vector normal.  
Los **vectores de soporte** son los puntos que *tensan la cuerda* del margen.

> Solo los vectores de soporte fijan la frontera; quitar puntos lejanos no la cambia.

<img src="im018.png" width="50%" style="display: block; margin-left: auto; margin-right: auto;">

## 3) Margen blando y el papel de **C**

En la práctica, los datos casi nunca son perfectamente separables. Por ello introducimos variables de holgura $\xi_i \geq 0$ que permiten ciertos errores:  

$$y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 - \xi_i.$$

El nuevo problema se convierte en:  

$$\min_{\mathbf{w}, b, \xi} \frac{1}{2} \| \mathbf{w} \|^2 + C \sum_i \xi_i,$$

donde el parámetro $C > 0$ controla el compromiso entre maximizar el margen y evitar errores de clasificación. Un $C$ grande penaliza con dureza los errores, prefiriendo fronteras ajustadas. Un $C$ pequeño permite más flexibilidad, buscando un margen más amplio aunque haya errores.

Datos reales = ruido + solapamientos. Permitimos holguras (errores) y regulamos con **C**:

- **C alto:** penaliza mucho los errores → frontera más rígida (riesgo de sobreajuste).  
- **C bajo:** permite más violaciones → frontera más suave (riesgo de subajuste).

<img src="im019.png" width="100%" style="display: block; margin-left: auto; margin-right: auto;">


## Cuando la separación lineal no alcanza: el truco del **kernel**
A veces, no hay ningún hiperplano lineal capaz de separar las clases.

Por ejemplo, imagina dos círculos concéntricos: uno con puntos amarillos en el centro y otro con puntos morados alrededor. Ninguna recta los separa.

La idea clave del SVM es **transformar el espacio original** mediante una función no lineal

$$\phi : \mathbb{R}^p \to \mathbb{R}^q,$$

llevando los datos a un espacio de mayor dimensión donde sí pueda existir una separación lineal.

Sin embargo, en lugar de calcular explícitamente $\phi(x)$, el algoritmo usa el *truco del kernel*:

$$K(x_i, x_j) = \langle \phi(x_i), \phi(x_j) \rangle$$

que permite calcular los productos internos en el espacio transformado sin necesidad de realizar la transformación.

<img src="im020.png" width="100%" style="display: block; margin-left: auto; margin-right: auto;">

### Kernels comunes

Cada kernel define un tipo de frontera diferente:

- Lineal:
  $$ K(x,z) = x^T z. $$
  $\rightarrow$ El modelo sigue siendo lineal.

- Polinomial:
  $$ K(x,z) = (x^T z + c)^d. $$
  $\rightarrow$ Permite fronteras curvas de distintos grados.

- RBF (Radial Basis Function o Gaussiano):
  $$ K(x,z) = \exp(-\gamma\|x - z\|^2). $$
  $\rightarrow$ Muy flexible, capaz de adaptarse a formas complejas.

- Sigmoide:
  $$ K(x,z) = \tanh(\alpha x^T z + c). $$
  $\rightarrow$ Inspirado en redes neuronales.

El parámetro $\gamma$ del kernel RBF controla el *radio de influencia* de cada punto: valores pequeños generan fronteras suaves; valores grandes producen fronteras muy onduladas y propensas al sobreajuste.

<img src="im021.png" width="100%" style="display: block; margin-left: auto; margin-right: auto;">

## De lo binario a lo multiclase

El algoritmo SVM original fue diseñado para dos clases.

Pero la extensión al caso multiclase ($k \geq 3$) se logra mediante estrategias de combinación de clasificadores binarios:

- **One-vs-Rest (OvR):** Se entrena un modelo para cada clase $i$, que la distingue frente a todas las demás. En predicción, se escoge la clase con la mayor puntuación.

- **One-vs-One (OvO):** Se entrena un modelo por cada par de clases $(i,j)$. En predicción, cada modelo emite un voto y gana la clase con más votos.

Ambas estrategias se usan en la práctica, y bibliotecas como ``scikit-learn``, permiten elegirlas de manera automática según la configuración del estimador.

<img src="im022.png" width="70%" style="display: block; margin-left: auto; margin-right: auto;">

## Regularización, C y margen blando

La regularización en SVM no se expresa con la típica norma $L_2$ añadida al costo (como en la regresión), sino que está implícita en el tamaño del margen.

El parámetro $C$ actúa como el regulador del equilibrio entre **ajuste** y **generalización**:

- $C \to \infty$: frontera muy rígida, riesgo de sobreajuste.
- $C \to 0$: frontera más laxa, riesgo de subajuste.

Visualmente, $C$ controla cuántos puntos se permiten dentro del margen o incluso mal clasificados.

## Interpretación geométrica final

El SVM puede entenderse como una máquina geométrica:

busca el hiperplano que deja la mayor distancia a los puntos más peligrosos (los vectores de soporte), como si estirara una cuerda entre ellos y la hiciera tensar al máximo.

En el caso multiclase, el espacio queda dividido en regiones donde domina la influencia de un subconjunto de esos vectores.

El resultado no es una simple frontera lineal, sino una colección de superficies suaves que equilibran márgenes entre cada par de clases.

## Buenas prácticas (conceptual)
- **Escalado previo.** SVM es sensible a escalas; conviene normalizar/estandarizar.  
- **Curación de outliers.** Pueden dominar el margen; revisar y limitar su impacto.  
- **Desbalance de clases.** Ajustar pesos de clase en el entrenamiento y cuidar la métrica de evaluación.  
- **Selección de kernel.** Lineal si la frontera parece simple o p ≫ n; RBF si esperas no linealidad.  
- **Validación.** Ajustar $C$ y $\gamma$ con validación rigurosa; evitar perseguir ruido.


## Resumen final
SVM (multiclase) = **fronteras con margen máximo**, con control de errores (**C**), y posibilidad de **no linealidad** vía **kernels** (controladas por **γ**).  
En multiclase, la coordinación OvR/OvO reparte el principio del margen entre todas las clases.

> Una máquina geométrica que prioriza “espacio libre” alrededor de la frontera para generalizar mejor.
